<a href="https://colab.research.google.com/github/shivrajshirsikar790-cmd/Stock-Price-Predictor/blob/main/02_Smart_Document_Q%26A%3B_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PDF Q&A Agent
Separate production-level Gemini notebook for college practical use.


## Installation
Use in Jupyter or Google Colab. Uncomment the install line if packages are missing.


In [4]:
!pip install -q google-genai python-dotenv tenacity pypdf


In [5]:
import os
import logging
from dataclasses import dataclass
from typing import List, Dict
from dotenv import load_dotenv
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from google import genai
from google.genai import types
from google.colab import userdata
from pypdf import PdfReader
from google.colab import files

In [6]:
load_dotenv()
GEMINI_API_KEY = userdata.get('Shivraj')  # use your secret name instead of assistant to access the api key .
if not GEMINI_API_KEY:
    raise ValueError('Missing GEMINI_API_KEY. Add it in .env or Colab environment settings.')

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('pdf-q-a-agent')
client = genai.Client(api_key=GEMINI_API_KEY)


In [7]:
@dataclass
class AgentConfig:
    name: str = 'PDF Q&A Agent'
    model: str = 'gemini-2.5-flash'
    temperature: float = 0.3
    max_output_tokens: int = 2048

SYSTEM_PROMPT = '''You are a PDF question-answering assistant. Answer based on provided extracted document text and say when information is missing.'''
config = AgentConfig()


In [8]:
class PdfQAAgent:
    def __init__(self, client: genai.Client, config: AgentConfig):
        self.client = client
        self.config = config

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10), retry=retry_if_exception_type(Exception))
    def generate(self, user_prompt: str, document_text: str | None = None, history: List[Dict[str, str]] | None = None) -> str:
        history = history or []
        contents = []

        # Add document text to the first user prompt if provided
        if document_text and not any('document_text' in item.get('content', '') for item in history if item.get('role') == 'user'):
            # This ensures document text is only added once at the beginning of the conversation
            initial_user_prompt = f"""Document:
{document_text}

Question: {user_prompt}"""
            contents.append(types.Content(role='user', parts=[types.Part(text=initial_user_prompt)]))
        else:
            # If no document text or it's already in history, append normally
            contents.append(types.Content(role='user', parts=[types.Part(text=user_prompt)]))

        # Add rest of the history
        for item in history:
            role = item.get('role', 'user')
            text = item.get('content', '')
            contents.append(types.Content(role=role, parts=[types.Part(text=text)]))

        response = self.client.models.generate_content(
            model=self.config.model,
            contents=contents,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=self.config.temperature,
                max_output_tokens=self.config.max_output_tokens
            )
        )
        return response.text.strip() if response.text else ''

In [9]:
agent = PdfQAAgent(client=client, config=config)


Now, let's create a function to handle PDF uploads, extract text, and then enable a Q&A session with the agent using that document.

In [10]:
def run_pdf_qa():
    uploaded = files.upload()
    pdf_content = ''

    if not uploaded:
        print("No file uploaded. Exiting PDF Q&A.")
        return

    for fn in uploaded.keys():
        print(f'User uploaded file "{fn}"')
        try:
            reader = PdfReader(fn)
            for page in reader.pages:
                pdf_content += page.extract_text() + "\n"
            print(f"Extracted {len(pdf_content)} characters from {fn}")
        except Exception as e:
            print(f"Error processing {fn}: {e}")
            return
        break # Process only the first uploaded PDF for simplicity

    if not pdf_content:
        print("Could not extract text from the PDF. Exiting.")
        return

    history = []
    print('\nPDF Q&A session started. Type quit to exit.')
    while True:
        user_input = input('You: ').strip()
        if user_input.lower() in {'quit', 'exit'}:
            print('Session ended.')
            break

        # Pass the extracted document text only with the first query or if not already in history
        reply = agent.generate(user_input, document_text=pdf_content, history=history)

        # Add user's question and agent's reply to history
        history.append({'role': 'user', 'content': user_input})
        history.append({'role': 'model', 'content': reply})

        print('Agent:', reply)  # use exit to end the session

# To start the PDF Q&A session:
run_pdf_qa()

Saving 1706.03762v7.pdf to 1706.03762v7.pdf
User uploaded file "1706.03762v7.pdf"
Extracted 39630 characters from 1706.03762v7.pdf

PDF Q&A session started. Type quit to exit.
You: what is transformers
Agent: The Transformer is a novel network architecture and a sequence transduction model that relies entirely on attention mechanisms, dispensing with recurrence and convolutions entirely. It uses stacked self-attention and point-wise, fully connected layers for both its encoder and decoder.

This architecture allows for significantly more parallelization and has achieved state-of-the-art results in translation quality.
You: quit
Session ended.


In [12]:
def run_chat():
    history = []
    print('Type quit to exit.')
    while True:
        user_input = input('You: ').strip()
        if user_input.lower() in {'quit', 'exit'}:
            print('Session ended.')
            break
        reply = agent.generate(user_input, history=history)
        history.append({'role': 'user', 'content': user_input})
        history.append({'role': 'model', 'content': reply})
        print('Agent:', reply)
